In [2]:
# ============================
# CORRECTED BATCH UNCERTAINTY QUANTIFICATION FOR ALL STATIONS
# ============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats as stats
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------- Configuration ----------
MAIN_RESULTS_DIR = "results_all_stations_uncertainty_corrected"
os.makedirs(MAIN_RESULTS_DIR, exist_ok=True)

# Define all station paths
STATION_PATHS = {
    "Aljarafe": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Aljarafe Results\results_dmoff_consistent_metrics",
    "Asomadilla": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Asomadilla Results\results_dmoff_consistent_metrics", 
    "Bermejales": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Bermejales Results\results_dmoff_consistent_metrics",
    "RonsadelVAalle": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\RonsadelVAalle Results\results_dmoff_consistent_metrics",
    "Torneo": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Torneo Results\results_dmoff_consistent_metrics"
}

print("=== CORRECTED BATCH UNCERTAINTY QUANTIFICATION FOR ALL STATIONS ===")
print(f"Processing {len(STATION_PATHS)} stations...")

# ---------- Corrected Core Uncertainty Functions ----------
def split_calibration_test(test_df, calibration_frac=0.3):
    """Split test data into calibration and evaluation sets to prevent data leakage"""
    # Use time-based split if possible, otherwise random
    if 'FECHA_HORA' in test_df.columns:
        test_df = test_df.sort_values('FECHA_HORA').reset_index(drop=True)
        calib_size = int(len(test_df) * calibration_frac)
        calib_df = test_df.iloc[:calib_size].copy()
        eval_df = test_df.iloc[calib_size:].copy()
        print(f"  Time-based split: Calibration={len(calib_df)}, Evaluation={len(eval_df)}")
    else:
        calib_df, eval_df = train_test_split(test_df, test_size=1-calibration_frac, random_state=RANDOM_SEED)
        print(f"  Random split: Calibration={len(calib_df)}, Evaluation={len(eval_df)}")
    
    return calib_df, eval_df

def calculate_calibrated_uncertainty(row, prediction, calibration_factor, tab_features):
    """Calculate properly calibrated uncertainty with realistic parameters"""
    # Base uncertainty from prediction magnitude (reduced)
    base_uncertainty = 0.08 * abs(prediction)
    
    # Feature-based uncertainty (reduced)
    if len(tab_features) > 0:
        feature_variability = np.std(row[tab_features].values.astype(float))
        feature_uncertainty = 0.05 * feature_variability
    else:
        feature_uncertainty = 0.5
    
    # Regime uncertainty (reduced)
    regime_uncertainty = 0.4
    
    # Residual uncertainty (more reasonable)
    residual_uncertainty = 1.5 * calibration_factor
    
    # Combined uncertainty (RMS)
    total_uncertainty = np.sqrt(base_uncertainty**2 + feature_uncertainty**2 + 
                               regime_uncertainty**2 + residual_uncertainty**2)
    
    return max(total_uncertainty, 1.0)  # Reduced minimum

def calibrated_ensemble_prediction(row, n_members, calibration_factor, tab_features):
    """Calibrated ensemble prediction with better uncertainty estimation"""
    ensemble_predictions = []
    
    for member in range(n_members):
        try:
            # Get base prediction from DMOFF
            base_pred = row['pred_dmoff']
            
            # Calculate IMPROVED uncertainty with calibration
            uncertainty = calculate_calibrated_uncertainty(row, base_pred, calibration_factor, tab_features)
            
            # Sample from calibrated predictive distribution
            final_pred = base_pred + np.random.normal(0, uncertainty)
            ensemble_predictions.append(final_pred)
            
        except Exception as e:
            ensemble_predictions.append(row['pred_dmoff'])
    
    return np.array(ensemble_predictions)

def calculate_calibrated_intervals(ensemble_predictions, confidence_level=0.90):
    """Calculate prediction intervals from ensemble"""
    alpha = (1 - confidence_level) / 2
    lower_percentile = alpha * 100
    upper_percentile = (1 - alpha) * 100
    
    mean_predictions = np.mean(ensemble_predictions, axis=1)
    std_predictions = np.std(ensemble_predictions, axis=1)
    lower_bounds = np.percentile(ensemble_predictions, lower_percentile, axis=1)
    upper_bounds = np.percentile(ensemble_predictions, upper_percentile, axis=1)
    
    return {
        'mean': mean_predictions,
        'std': std_predictions,
        'lower': lower_bounds,
        'upper': upper_bounds,
        'interval_width': upper_bounds - lower_bounds
    }

def proper_conformal_prediction_calibration(calib_df, eval_df, confidence_level=0.90):
    """Proper conformal prediction with separate calibration set to prevent data leakage"""
    # Calculate residuals on calibration set only
    residuals = np.abs(calib_df['obs'] - calib_df['pred_dmoff'])
    
    # Get conformal quantile from calibration set
    alpha = 1 - confidence_level
    conformal_quantile = np.quantile(residuals, 1 - alpha)
    
    # Apply to evaluation set only
    eval_df_out = eval_df.copy()
    eval_df_out['pred_lower_conformal'] = eval_df_out['pred_dmoff'] - conformal_quantile
    eval_df_out['pred_upper_conformal'] = eval_df_out['pred_dmoff'] + conformal_quantile
    eval_df_out['interval_width_conformal'] = 2 * conformal_quantile
    
    return eval_df_out, conformal_quantile

def proper_quantile_regression(calib_df, eval_df, features, target='obs', quantiles=[0.05, 0.5, 0.95]):
    """Quantile regression trained on calibration set, evaluated on test set to prevent data leakage"""
    X_calib = calib_df[features].values
    y_calib = calib_df[target].values
    X_eval = eval_df[features].values
    
    quantile_predictions = {}
    
    for q in quantiles:
        if q == 0.5:
            model = GradientBoostingRegressor(
                loss='absolute_error', 
                n_estimators=100,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                random_state=RANDOM_SEED
            )
        else:
            model = GradientBoostingRegressor(
                loss='quantile', 
                alpha=q,
                n_estimators=100,
                max_depth=4, 
                learning_rate=0.05,
                subsample=0.8,
                random_state=RANDOM_SEED
            )
        
        model.fit(X_calib, y_calib)
        quantile_predictions[q] = model.predict(X_eval)
    
    return quantile_predictions

def calculate_coverage_metrics(y_true, y_lower, y_upper):
    """Calculate coverage metrics for prediction intervals"""
    coverage = np.mean((y_true >= y_lower) & (y_true <= y_upper))
    avg_width = np.mean(y_upper - y_lower)
    calibration_error = abs(coverage - 0.90)
    
    return {
        'coverage': coverage,
        'avg_interval_width': avg_width,
        'calibration_error': calibration_error
    }

def calculate_all_calibration_metrics(test_df):
    """Calculate calibration metrics for all methods"""
    metrics = {}
    
    # Calibrated Ensemble
    metrics['calibrated_ensemble'] = calculate_coverage_metrics(
        test_df['obs'], test_df['pred_lower_calibrated'], test_df['pred_upper_calibrated']
    )
    
    # Conformal Prediction
    metrics['conformal'] = calculate_coverage_metrics(
        test_df['obs'], test_df['pred_lower_conformal'], test_df['pred_upper_conformal']
    )
    
    # Improved Quantile
    metrics['improved_quantile'] = calculate_coverage_metrics(
        test_df['obs'], test_df['pred_lower_improved_q'], test_df['pred_upper_improved_q'] 
    )
    
    return metrics

# ---------- Corrected Main Processing Function ----------
def process_station_uncertainty_corrected(station_name, station_path):
    """Process uncertainty quantification for a single station with data leakage fixes"""
    print(f"\n{'='*60}")
    print(f"PROCESSING STATION: {station_name}")
    print(f"{'='*60}")
    
    # Create station-specific results directory
    station_uq_dir = os.path.join(MAIN_RESULTS_DIR, f"UQ_{station_name}")
    os.makedirs(station_uq_dir, exist_ok=True)
    
    try:
        # Step 1: Load Data and Models
        print("Step 1: Loading DMOFF artifacts...")
        test_df = pd.read_csv(os.path.join(station_path, "test_predictions_dmoff.csv"))
        print(f"Loaded test data: {len(test_df)} samples")
        
        # Load models (handle potential missing files gracefully)
        regime_models = None
        rfc = None
        clf_scaler = None
        
        try:
            regime_models = joblib.load(os.path.join(station_path, "regime_models.pkl"))
            rfc = joblib.load(os.path.join(station_path, "regime_classifier_rf.pkl"))
            clf_scaler = joblib.load(os.path.join(station_path, "regime_clf_scaler.pkl"))
            print("✓ Successfully loaded all DMOFF artifacts")
        except Exception as e:
            print(f"⚠ Could not load some models: {e}")
            print("Proceeding with basic uncertainty quantification...")
        
        # Step 2: Define Feature Columns
        print("Step 2: Defining feature columns...")
        tab_features = [col for col in test_df.columns if any(x in col for x in ['_lag', 'hour', 'month', 'weekday', 'CO', 'DD', 'NO2', 'PM10', 'TMP', 'VV'])
                       and col not in ['obs', 'pred_dmoff', 'regime', 'FECHA_HORA', 'error']]
        clf_features = [col for col in test_df.columns if col in ['CO', 'DD', 'NO2', 'PM10', 'TMP', 'VV', 'O3', 'O3_lag1', 'O3_lag2', 'O3_lag3', 'O3_lag24',
                       'hour_sin', 'hour_cos', 'month_sin', 'month_cos'] and col not in ['obs', 'pred_dmoff']]
        
        print(f"✓ Features defined: {len(tab_features)} tabular, {len(clf_features)} classifier")
        
        # Step 3: Split data into calibration and evaluation sets (CRITICAL FIX)
        print("Step 3: Splitting data into calibration and evaluation sets...")
        calib_df, eval_df = split_calibration_test(test_df, calibration_frac=0.3)
        print(f"✓ Data split: {len(calib_df)} calibration samples, {len(eval_df)} evaluation samples")
        
        # Step 4: Generate CALIBRATED Ensemble (on evaluation set only)
        print("Step 4: Generating calibrated ensemble predictions...")
        calibrated_ensemble_predictions = []
        
        for idx, row in eval_df.iterrows():
            if idx % 100 == 0 and idx > 0:
                print(f"  Processing sample {idx}/{len(eval_df)}")
            
            # Use more reasonable calibration factor
            ensemble_preds = calibrated_ensemble_prediction(row, n_members=100, calibration_factor=1.2, tab_features=tab_features)
            calibrated_ensemble_predictions.append(ensemble_preds)
        
        calibrated_ensemble_predictions = np.array(calibrated_ensemble_predictions)
        print(f"✓ Calibrated ensemble generated: {calibrated_ensemble_predictions.shape}")
        
        # Step 5: Calculate Intervals
        calibrated_intervals = calculate_calibrated_intervals(calibrated_ensemble_predictions)
        
        # Update evaluation dataframe
        eval_df['pred_mean_calibrated'] = calibrated_intervals['mean']
        eval_df['pred_std_calibrated'] = calibrated_intervals['std']
        eval_df['pred_lower_calibrated'] = calibrated_intervals['lower']
        eval_df['pred_upper_calibrated'] = calibrated_intervals['upper']
        eval_df['interval_width_calibrated'] = calibrated_intervals['interval_width']
        
        # Step 6: PROPER CONFORMAL PREDICTION CALIBRATION (CRITICAL FIX)
        print("Step 6: Applying proper conformal prediction calibration...")
        eval_df, conformal_quantile = proper_conformal_prediction_calibration(calib_df, eval_df)
        print(f"Conformal quantile (non-conformity score): {conformal_quantile:.3f}")
        
        # Step 7: PROPER Quantile Regression (CRITICAL FIX)
        print("Step 7: Training proper quantile regression...")
        if len(tab_features) > 0 and len(calib_df) > 50:  # Ensure enough calibration samples
            improved_quantile_preds = proper_quantile_regression(calib_df, eval_df, tab_features)
            
            eval_df['pred_lower_improved_q'] = improved_quantile_preds[0.05]
            eval_df['pred_upper_improved_q'] = improved_quantile_preds[0.95]
            eval_df['pred_median_improved_q'] = improved_quantile_preds[0.5]
            eval_df['interval_width_improved_q'] = eval_df['pred_upper_improved_q'] - eval_df['pred_lower_improved_q']
        else:
            print("⚠ Not enough samples or features for quantile regression")
            # Create reasonable fallback
            eval_df['pred_lower_improved_q'] = eval_df['pred_lower_conformal']
            eval_df['pred_upper_improved_q'] = eval_df['pred_upper_conformal']
            eval_df['pred_median_improved_q'] = eval_df['pred_dmoff']
            eval_df['interval_width_improved_q'] = eval_df['interval_width_conformal']
        
        # Step 8: Calculate ALL Calibration Metrics (on evaluation set only)
        print("Step 8: Calculating calibration metrics...")
        all_metrics = calculate_all_calibration_metrics(eval_df)
        
        # Print results
        print(f"\n{'='*50}")
        print(f"CALIBRATION RESULTS - {station_name}")
        print(f"{'='*50}")
        
        for method, metrics in all_metrics.items():
            print(f"\n{method.upper().replace('_', ' ')}:")
            print(f"  Coverage: {metrics['coverage']:.3f} (Target: 0.900)")
            print(f"  Average Width: {metrics['avg_interval_width']:.3f}")
            print(f"  Calibration Error: {metrics['calibration_error']:.3f}")
        
        # Step 9: Create Visualizations
        print("Step 9: Creating visualizations...")
        plt.figure(figsize=(15, 10))
        
        # 1. Comparison of all methods
        plt.subplot(2, 2, 1)
        methods = list(all_metrics.keys())
        coverages = [all_metrics[m]['coverage'] for m in methods]
        widths = [all_metrics[m]['avg_interval_width'] for m in methods]
        
        x = np.arange(len(methods))
        width = 0.35
        
        plt.bar(x - width/2, coverages, width, label='Coverage', alpha=0.8, color='lightblue')
        plt.bar(x + width/2, [w/50 for w in widths], width, label='Width/50', alpha=0.8, color='lightcoral')
        plt.axhline(y=0.9, color='red', linestyle='--', label='Target Coverage', linewidth=2)
        plt.xlabel('Method')
        plt.ylabel('Coverage / Scaled Width')
        plt.title(f'UQ Methods Comparison - {station_name}')
        plt.xticks(x, [m.replace('_', '\n').title() for m in methods])
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 2. Time series with BEST method intervals
        best_method = min(all_metrics.items(), key=lambda x: x[1]['calibration_error'])[0]
        
        if best_method == 'calibrated_ensemble':
            lower_col, upper_col = 'pred_lower_calibrated', 'pred_upper_calibrated'
            mean_col = 'pred_mean_calibrated'
        elif best_method == 'conformal':
            lower_col, upper_col = 'pred_lower_conformal', 'pred_upper_conformal' 
            mean_col = 'pred_dmoff'
        else:
            lower_col, upper_col = 'pred_lower_improved_q', 'pred_upper_improved_q'
            mean_col = 'pred_median_improved_q'
        
        plt.subplot(2, 2, 2)
        # Plot only first 200 points for clarity
        n_plot = min(200, len(eval_df))
        plt.plot(range(n_plot), eval_df['obs'].values[:n_plot], 'k-', alpha=0.8, label='Observed', linewidth=1)
        plt.plot(range(n_plot), eval_df[mean_col].values[:n_plot], 'b-', alpha=0.8, label='Predicted', linewidth=1)
        plt.fill_between(range(n_plot), eval_df[lower_col].values[:n_plot], eval_df[upper_col].values[:n_plot], 
                        alpha=0.3, color='blue', label='90% Prediction Interval')
        plt.xlabel('Sample Index')
        plt.ylabel('O3 Concentration')
        plt.title(f'Best Method: {best_method.replace("_", " ").title()}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 3. Calibration plot for best method
        plt.subplot(2, 2, 3)
        nominal_coverage = np.linspace(0.05, 0.95, 19)
        empirical_coverage = []
        
        if best_method == 'calibrated_ensemble':
            ensemble_preds = calibrated_ensemble_predictions
        else:
            # For conformal and quantile, we need to simulate
            ensemble_preds = np.random.normal(
                eval_df[mean_col].values[:, None], 
                eval_df[mean_col].std(), 
                (len(eval_df), 100)
            )
        
        for level in nominal_coverage:
            lower = np.percentile(ensemble_preds, (1-level)/2 * 100, axis=1)
            upper = np.percentile(ensemble_preds, (1+level)/2 * 100, axis=1)
            coverage = np.mean((eval_df['obs'] >= lower) & (eval_df['obs'] <= upper))
            empirical_coverage.append(coverage)
        
        plt.plot(nominal_coverage, empirical_coverage, 'bo-', label='Empirical Coverage', markersize=3)
        plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)
        plt.xlabel('Nominal Coverage')
        plt.ylabel('Empirical Coverage')
        plt.title('Calibration Plot')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 4. Interval width vs error
        plt.subplot(2, 2, 4)
        errors = np.abs(eval_df['obs'] - eval_df[mean_col])
        interval_widths = eval_df[upper_col] - eval_df[lower_col]
        
        plt.scatter(interval_widths, errors, alpha=0.6, s=20)
        plt.xlabel('Prediction Interval Width')
        plt.ylabel('Absolute Error')
        plt.title('Interval Width vs Prediction Error')
        
        # Calculate and display correlation
        if len(interval_widths) > 1 and np.std(interval_widths) > 0 and np.std(errors) > 0:
            correlation = np.corrcoef(interval_widths, errors)[0, 1]
        else:
            correlation = 0.0
        
        plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
                 transform=plt.gca().transAxes, bbox=dict(boxstyle="round", facecolor='wheat', alpha=0.8))
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(station_uq_dir, f"uncertainty_analysis_{station_name}.png"), 
                    dpi=300, bbox_inches='tight')
        plt.close()  # Close plot to free memory
        
        # Step 10: Save Station Results
        print("Step 10: Saving station results...")
        
        # Save evaluation data with all predictions
        eval_df.to_csv(os.path.join(station_uq_dir, f"calibrated_predictions_{station_name}.csv"), index=False)
        
        # Calculate performance metrics
        rmse_calibrated = float(np.sqrt(mean_squared_error(eval_df['obs'], eval_df['pred_mean_calibrated'])))
        rmse_conformal = float(np.sqrt(mean_squared_error(eval_df['obs'], eval_df['pred_dmoff'])))  # Use original DMOFF
        rmse_quantile = float(np.sqrt(mean_squared_error(eval_df['obs'], eval_df['pred_median_improved_q'])))
        
        # Create comprehensive station summary
        station_summary = {
            'station': station_name,
            'n_samples_total': len(test_df),
            'n_samples_calibration': len(calib_df),
            'n_samples_evaluation': len(eval_df),
            'best_method': best_method,
            'conformal_quantile': float(conformal_quantile),
            'calibrated_ensemble_coverage': all_metrics['calibrated_ensemble']['coverage'],
            'calibrated_ensemble_width': all_metrics['calibrated_ensemble']['avg_interval_width'],
            'calibrated_ensemble_calibration_error': all_metrics['calibrated_ensemble']['calibration_error'],
            'calibrated_ensemble_rmse': rmse_calibrated,
            'conformal_coverage': all_metrics['conformal']['coverage'],
            'conformal_width': all_metrics['conformal']['avg_interval_width'],
            'conformal_calibration_error': all_metrics['conformal']['calibration_error'],
            'conformal_rmse': rmse_conformal,
            'quantile_coverage': all_metrics['improved_quantile']['coverage'],
            'quantile_width': all_metrics['improved_quantile']['avg_interval_width'],
            'quantile_calibration_error': all_metrics['improved_quantile']['calibration_error'],
            'quantile_rmse': rmse_quantile
        }
        
        # Save station summary
        with open(os.path.join(station_uq_dir, f"station_summary_{station_name}.json"), "w") as f:
            json.dump(station_summary, f, indent=2)
        
        print(f"✓ Station {station_name} processing completed!")
        return station_summary
        
    except Exception as e:
        print(f"❌ Error processing station {station_name}: {e}")
        import traceback
        traceback.print_exc()
        # Return minimal summary for failed station
        return {
            'station': station_name,
            'n_samples_total': 0,
            'n_samples_calibration': 0,
            'n_samples_evaluation': 0,
            'best_method': 'failed',
            'error': str(e)
        }

# ---------- Main Execution ----------
def main():
    """Main function to process all stations"""
    print("Starting corrected batch uncertainty quantification...")
    print(f"Processing {len(STATION_PATHS)} stations")
    
    all_station_summaries = []
    
    # Process each station
    for station_name, station_path in STATION_PATHS.items():
        if os.path.exists(station_path):
            station_summary = process_station_uncertainty_corrected(station_name, station_path)
            all_station_summaries.append(station_summary)
        else:
            print(f"❌ Station path not found: {station_path}")
            all_station_summaries.append({
                'station': station_name,
                'n_samples_total': 0,
                'n_samples_calibration': 0,
                'n_samples_evaluation': 0,
                'best_method': 'path_not_found',
                'error': f"Path {station_path} does not exist"
            })
    
    # Create Comprehensive Paper-Ready Table
    print(f"\n{'='*80}")
    print("CREATING COMPREHENSIVE PAPER-READY RESULTS TABLE")
    print(f"{'='*80}")
    
    # Convert to DataFrame
    summary_df = pd.DataFrame(all_station_summaries)
    
    # Create paper-ready comparison table
    paper_table = []
    
    for station in summary_df.to_dict('records'):
        if 'error' not in station or station['error'] == 'path_not_found':
            paper_table.append({
                'Station': station['station'],
                'Samples_Total': station.get('n_samples_total', 0),
                'Samples_Eval': station.get('n_samples_evaluation', 0),
                'Best_Method': station.get('best_method', 'N/A').replace('_', ' ').title(),
                'Calibrated_Ensemble_Coverage': f"{station.get('calibrated_ensemble_coverage', 0):.3f}",
                'Calibrated_Ensemble_Width': f"{station.get('calibrated_ensemble_width', 0):.2f}",
                'Calibrated_Ensemble_Calib_Error': f"{station.get('calibrated_ensemble_calibration_error', 0):.3f}",
                'Conformal_Coverage': f"{station.get('conformal_coverage', 0):.3f}",
                'Conformal_Width': f"{station.get('conformal_width', 0):.2f}",
                'Conformal_Calib_Error': f"{station.get('conformal_calibration_error', 0):.3f}",
                'Quantile_Coverage': f"{station.get('quantile_coverage', 0):.3f}",
                'Quantile_Width': f"{station.get('quantile_width', 0):.2f}",
            })
        else:
            paper_table.append({
                'Station': station['station'],
                'Samples_Total': 'Failed',
                'Samples_Eval': 'Failed',
                'Best_Method': 'Processing Error',
                'Calibrated_Ensemble_Coverage': 'N/A',
                'Calibrated_Ensemble_Width': 'N/A',
                'Calibrated_Ensemble_Calib_Error': 'N/A',
                'Conformal_Coverage': 'N/A',
                'Conformal_Width': 'N/A',
                'Conformal_Calib_Error': 'N/A',
                'Quantile_Coverage': 'N/A',
                'Quantile_Width': 'N/A',
            })
    
    # Create final paper table
    paper_df = pd.DataFrame(paper_table)
    
    # Save comprehensive results
    paper_df.to_csv(os.path.join(MAIN_RESULTS_DIR, "PAPER_READY_ALL_STATIONS_UQ_TABLE_CORRECTED.csv"), index=False)
    summary_df.to_csv(os.path.join(MAIN_RESULTS_DIR, "detailed_all_stations_summary_corrected.csv"), index=False)
    
    # Print beautiful final table
    print("\n" + "="*120)
    print("FINAL CORRECTED UNCERTAINTY QUANTIFICATION RESULTS")
    print("="*120)
    print("\nCoverage Target: 0.900 | Lower Calibration Error = Better")
    print("-" * 120)
    
    # Display formatted table
    display_columns = ['Station', 'Samples_Eval', 'Best_Method', 'Calibrated_Ensemble_Coverage', 
                      'Calibrated_Ensemble_Calib_Error', 'Conformal_Coverage', 'Conformal_Calib_Error', 'Quantile_Coverage']
    
    display_df = paper_df[display_columns].copy()
    display_df.columns = ['Station', 'Eval Samples', 'Best Method', 'Calib Ens Cov', 
                         'Calib Ens Err', 'Conformal Cov', 'Conformal Err', 'Quantile Cov']
    
    print(display_df.to_string(index=False))
    
    # Calculate overall statistics
    successful_stations = [s for s in all_station_summaries if s.get('n_samples_evaluation', 0) > 0]
    if successful_stations:
        avg_calibrated_coverage = np.mean([s.get('calibrated_ensemble_coverage', 0) for s in successful_stations])
        avg_conformal_coverage = np.mean([s.get('conformal_coverage', 0) for s in successful_stations])
        avg_quantile_coverage = np.mean([s.get('quantile_coverage', 0) for s in successful_stations])
        
        avg_calibrated_error = np.mean([s.get('calibrated_ensemble_calibration_error', 0) for s in successful_stations])
        avg_conformal_error = np.mean([s.get('conformal_calibration_error', 0) for s in successful_stations])
        avg_quantile_error = np.mean([s.get('quantile_calibration_error', 0) for s in successful_stations])
        
        print(f"\n{'='*80}")
        print("OVERALL SUMMARY STATISTICS (Successful Stations Only)")
        print(f"{'='*80}")
        print(f"Number of successful stations: {len(successful_stations)}/{len(STATION_PATHS)}")
        print(f"Average Calibrated Ensemble: Coverage={avg_calibrated_coverage:.3f}, Error={avg_calibrated_error:.3f}")
        print(f"Average Conformal Prediction: Coverage={avg_conformal_coverage:.3f}, Error={avg_conformal_error:.3f}")
        print(f"Average Quantile Regression: Coverage={avg_quantile_coverage:.3f}, Error={avg_quantile_error:.3f}")
        
        # Recommendation based on calibration error
        best_overall_method = min(['calibrated_ensemble', 'conformal', 'quantile'], 
                                 key=lambda m: np.mean([s.get(f'{m}_calibration_error', 1) for s in successful_stations]))
        
        print(f"\n🎯 RECOMMENDED METHOD FOR PAPER: {best_overall_method.upper().replace('_', ' ')}")
        print(f"   Average Coverage: {np.mean([s.get(f'{best_overall_method}_coverage', 0) for s in successful_stations]):.3f}")
        print(f"   Average Calibration Error: {np.mean([s.get(f'{best_overall_method}_calibration_error', 0) for s in successful_stations]):.3f}")
    
    print(f"\n📁 All results saved to: {MAIN_RESULTS_DIR}")
    print("✅ CORRECTED BATCH PROCESSING COMPLETED SUCCESSFULLY!")
    print("="*120)

# ---------- Run Main Function ----------
if __name__ == "__main__":
    main()

=== CORRECTED BATCH UNCERTAINTY QUANTIFICATION FOR ALL STATIONS ===
Processing 5 stations...
Starting corrected batch uncertainty quantification...
Processing 5 stations

PROCESSING STATION: Aljarafe
Step 1: Loading DMOFF artifacts...
Loaded test data: 8511 samples
✓ Successfully loaded all DMOFF artifacts
Step 2: Defining feature columns...
✓ Features defined: 36 tabular, 14 classifier
Step 3: Splitting data into calibration and evaluation sets...
  Time-based split: Calibration=2553, Evaluation=5958
✓ Data split: 2553 calibration samples, 5958 evaluation samples
Step 4: Generating calibrated ensemble predictions...
  Processing sample 2600/5958
  Processing sample 2700/5958
  Processing sample 2800/5958
  Processing sample 2900/5958
  Processing sample 3000/5958
  Processing sample 3100/5958
  Processing sample 3200/5958
  Processing sample 3300/5958
  Processing sample 3400/5958
  Processing sample 3500/5958
  Processing sample 3600/5958
  Processing sample 3700/5958
  Processing sa